# Notebook for merging the datasets

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
#mouting the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd drive/MyDrive/TFG/oulad

/content/drive/MyDrive/TFG/oulad


In [4]:
vle = pd.read_csv('vle.csv')
studentVle = pd.read_csv('studentVle.csv')
studentRegistration = pd.read_csv('studentRegistration.csv')
studentInfo = pd.read_csv('studentInfo.csv')
studentAssessment = pd.read_csv('studentAssessment.csv')
courses = pd.read_csv('courses.csv')
assessments = pd.read_csv('assessments.csv')

In [5]:
def condition(x):
    if x=='CCC':
      return 0
    if x=='DDD':
      return 0
    else:
      return

In [6]:
date1=15

In [7]:
studentVle1= studentVle[studentVle['date'] <= date1]
studentAssessment1= studentAssessment[studentAssessment['date_submitted'] <= date1]
df_student=pd.merge(studentRegistration, studentInfo, on=['id_student', 'code_module', 'code_presentation'])
df_vle=pd.merge(vle, studentVle1, on=['id_site', 'code_module', 'code_presentation'])
df_vle_mean=df_vle.groupby(['id_student', 'code_module', 'code_presentation']).sum_click.mean().reset_index(name='mean_clicks')
df_vle_sum=df_vle.groupby(['id_student', 'code_module', 'code_presentation']).sum_click.sum().reset_index(name='total_clicks')
df_vle_mean_by_date=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date']).sum_click.mean().reset_index(name='mean_clicks')
df_vle_sum_by_date=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date']).sum_click.sum().reset_index(name='sum_clicks')
df_vle_sum_type=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'activity_type']).sum_click.sum().reset_index(name='total_clicks')
activities=list(df_vle_sum_type.activity_type.unique())
for activity in activities:
    df_sum_x=df_vle_sum_type[df_vle_sum_type.activity_type==activity].drop('activity_type', axis=1).rename(columns={'total_clicks': 'clicks_from_' + str(activity)})
    df_vle_sum=pd.merge(df_vle_sum, df_sum_x, on =['id_student', 'code_module', 'code_presentation'], how = 'outer')
df_vle_sum_by_date_type=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date', 'activity_type']).sum_click.sum().reset_index(name='sum_clicks')
activities=list(df_vle_sum_by_date_type.activity_type.unique())
for activity in activities:    
  df_sum_x=df_vle_sum_by_date_type[df_vle_sum_by_date_type.activity_type==activity].drop('activity_type', axis=1).rename(columns={'sum_clicks': 'clicks_from_' + str(activity)})
  df_vle_sum_by_date=pd.merge(df_vle_sum_by_date, df_sum_x, on =['id_student', 'code_module', 'code_presentation', 'date'], how = 'outer')
assessments.loc[(assessments['code_module'] == 'GGG') & (assessments['assessment_type'] != 'Exam'), 'weight']=11
df_assessments=pd.merge(studentAssessment1, assessments, on=['id_assessment'])
df_assesments_score_mean=df_assessments.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='mean_score')

df_assessments_tma=df_assessments[(df_assessments['assessment_type']=='TMA')]
df_assesments_tma_score_mean=df_assessments_tma.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='tma_mean_score')
df_assessments_cma=df_assessments[(df_assessments['assessment_type']=='CMA')]
df_assesments_cma_score_mean=df_assessments_cma.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='cma_mean_score')
df_assessments_exam=df_assessments[(df_assessments['assessment_type']=='Exam')]
df_assesments_exam_score_mean=df_assessments_exam.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='exam_mean_score')
df_score_mean_1=pd.merge(df_assesments_tma_score_mean, df_assesments_cma_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
df_score_mean=pd.merge(df_score_mean_1, df_assesments_exam_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
df_assessments_mean=pd.merge(df_score_mean, df_assesments_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
df_final_grade=df_assessments[['id_student', 'score', 'code_module','code_presentation', 'assessment_type', 'weight']].copy()
df_final_grade['weighted']=df_final_grade.score * df_final_grade.weight * 0.01
df_final_grade_cont=df_final_grade[(df_final_grade['assessment_type']!='Exam')]
df_final_grade_cont=df_final_grade_cont.groupby(['id_student', 'code_module', 'code_presentation']).weighted.sum().reset_index(name='continuous_grade')
df_final_grade_exam=df_final_grade[(df_final_grade['assessment_type']=='Exam')]
df_final_grade_exam=df_final_grade_exam.groupby(['id_student', 'code_module', 'code_presentation']).weighted.sum().reset_index(name='exam_grade')
df_final=df_student.copy()
df_final=pd.merge(df_final, df_assessments_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
df_final=pd.merge(df_final, df_vle_sum, on=['id_student', 'code_module', 'code_presentation'], how="outer")
df_final=pd.merge(df_final, df_vle_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
df_final=pd.merge(df_final, df_final_grade_cont, on=['id_student', 'code_module', 'code_presentation'], how="outer")

df_final=pd.merge(df_final, df_final_grade_exam, on=['id_student', 'code_module', 'code_presentation'], how="outer")
df_final=pd.merge(df_final, courses, on=['code_module', 'code_presentation'], how="outer")
df_final['mean_clicks_per_day']=df_final['mean_clicks']/df_final['module_presentation_length']
df_final['date_registration'] = df_final['date_registration'].fillna(0)
df_final.loc[pd.isna(df_final["date_unregistration"]), "date_unregistration"] = df_final[pd.isna(df_final["date_unregistration"])]["module_presentation_length"].apply(
    lambda x: x
)

df_final['date_registration'] = df_final['date_registration'].fillna(0)
df_final.iloc[:,17:41]=df_final.iloc[:,17:41].fillna(0)
df_final.loc[pd.isna(df_final["exam_grade"]), "exam_grade"] = df_final[pd.isna(df_final["exam_grade"])]["code_module"].apply(condition)
df_final.loc[pd.isna(df_final["exam_grade"]), "exam_grade"] = df_final[pd.isna(df_final["exam_grade"])]["continuous_grade"].apply(lambda x: x)
df_final['mean_clicks_per_day'] = df_final['mean_clicks_per_day'].fillna(0)
df_final.drop(['tma_mean_score', 'cma_mean_score', 'exam_mean_score'], axis=1, inplace=True)
df_final['Date']=date1

In [8]:
df_full=df_final.copy()

In [9]:
df_full

,code_module,code_presentation,id_student,date_registration,date_unregistration,gender,region,highest_education,imd_band,age_band,...,clicks_from_ouelluminate,clicks_from_htmlactivity,clicks_from_questionnaire,clicks_from_sharedsubpage,mean_clicks,continuous_grade,exam_grade,module_presentation_length,mean_clicks_per_day,Date
0,AAA,2013J,11391,-159.0,268.0,M,East Anglian Region,HE Qualification,90-100%,55<=,...,0.0,0.0,0.0,0.0,6.688889,0.0,0.0,268,0.024959,15
1,AAA,2013J,28400,-53.0,268.0,F,Scotland,HE Qualification,20-30%,35-55,...,0.0,0.0,0.0,0.0,3.648000,0.0,0.0,268,0.013612,15
2,AAA,2013J,30268,-92.0,12.0,F,North Western Region,A Level or Equivalent,30-40%,35-55,...,0.0,0.0,0.0,0.0,3.697368,0.0,0.0,268,0.013796,15
3,AAA,2013J,31604,-52.0,268.0,F,South East Region,A Level or Equivalent,50-60%,35-55,...,0.0,0.0,0.0,0.0,3.835165,0.0,0.0,268,0.014310,15
4,AAA,2013J,32885,-176.0,268.0,F,West Midlands Region,Lower Than A Level,50-60%,0-35,...,0.0,0.0,0.0,0.0,4.140351,0.0,0.0,268,0.015449,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32588,GGG,2014J,2640965,-4.0,269.0,F,Wales,Lower Than A Level,10-20,0-35,...,0.0,0.0,0.0,0.0,3.166667,0.0,0.0,269,0.011772,15
32589,GGG,2014J,2645731,-23.0,269.0,F,East Anglian Region,Lower Than A Level,40-50%,35-55,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,269,0.000000,15
32590,GGG,2014J,2648187,-129.0,269.0,F,South Region,A Level or Equivalent,20-30%,0-35,...,0.0,0.0,0.0,0.0,1.600000,0.0,0.0,269,0.005948,15
32591,GGG,2014J,2679821,-49.0,101.0,F,South East Region,Lower Than A Level,90-100%,35-55,...,0.0,0.0,0.0,0.0,3.000000,0.0,0.0,269,0.011152,15


In [10]:
#dates=[30,45,60, 75, 90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240, 255, 270]

In [16]:
dates=[240]

In [17]:
for date in dates:
  studentVle1= studentVle[studentVle['date'] <= date]
  studentAssessment1= studentAssessment[studentAssessment['date_submitted'] <= date]
  df_student=pd.merge(studentRegistration, studentInfo, on=['id_student', 'code_module', 'code_presentation'])
  df_vle=pd.merge(vle, studentVle1, on=['id_site', 'code_module', 'code_presentation'])
  df_vle_mean=df_vle.groupby(['id_student', 'code_module', 'code_presentation']).sum_click.mean().reset_index(name='mean_clicks')
  df_vle_sum=df_vle.groupby(['id_student', 'code_module', 'code_presentation']).sum_click.sum().reset_index(name='total_clicks')
  df_vle_mean_by_date=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date']).sum_click.mean().reset_index(name='mean_clicks')
  df_vle_sum_by_date=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date']).sum_click.sum().reset_index(name='sum_clicks')
  df_vle_sum_type=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'activity_type']).sum_click.sum().reset_index(name='total_clicks')
  activities=list(df_vle_sum_type.activity_type.unique())
  for activity in activities:
      df_sum_x=df_vle_sum_type[df_vle_sum_type.activity_type==activity].drop('activity_type', axis=1).rename(columns={'total_clicks': 'clicks_from_' + str(activity)})
      df_vle_sum=pd.merge(df_vle_sum, df_sum_x, on =['id_student', 'code_module', 'code_presentation'], how = 'outer')
  df_vle_sum_by_date_type=df_vle.groupby(['id_student', 'code_module', 'code_presentation', 'date', 'activity_type']).sum_click.sum().reset_index(name='sum_clicks')
  activities=list(df_vle_sum_by_date_type.activity_type.unique())
  for activity in activities:    
    df_sum_x=df_vle_sum_by_date_type[df_vle_sum_by_date_type.activity_type==activity].drop('activity_type', axis=1).rename(columns={'sum_clicks': 'clicks_from_' + str(activity)})
    df_vle_sum_by_date=pd.merge(df_vle_sum_by_date, df_sum_x, on =['id_student', 'code_module', 'code_presentation', 'date'], how = 'outer')
  assessments.loc[(assessments['code_module'] == 'GGG') & (assessments['assessment_type'] != 'Exam'), 'weight']=11
  df_assessments=pd.merge(studentAssessment1, assessments, on=['id_assessment'])
  df_assesments_score_mean=df_assessments.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='mean_score')

  df_assessments_tma=df_assessments[(df_assessments['assessment_type']=='TMA')]
  df_assesments_tma_score_mean=df_assessments_tma.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='tma_mean_score')
  df_assessments_cma=df_assessments[(df_assessments['assessment_type']=='CMA')]
  df_assesments_cma_score_mean=df_assessments_cma.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='cma_mean_score')
  df_assessments_exam=df_assessments[(df_assessments['assessment_type']=='Exam')]
  df_assesments_exam_score_mean=df_assessments_exam.groupby(['id_student', 'code_module', 'code_presentation']).score.mean().reset_index(name='exam_mean_score')
  df_score_mean_1=pd.merge(df_assesments_tma_score_mean, df_assesments_cma_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
  df_score_mean=pd.merge(df_score_mean_1, df_assesments_exam_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
  df_assessments_mean=pd.merge(df_score_mean, df_assesments_score_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
  df_final_grade=df_assessments[['id_student', 'score', 'code_module','code_presentation', 'assessment_type', 'weight']].copy()
  df_final_grade['weighted']=df_final_grade.score * df_final_grade.weight * 0.01
  df_final_grade_cont=df_final_grade[(df_final_grade['assessment_type']!='Exam')]
  df_final_grade_cont=df_final_grade_cont.groupby(['id_student', 'code_module', 'code_presentation']).weighted.sum().reset_index(name='continuous_grade')
  df_final_grade_exam=df_final_grade[(df_final_grade['assessment_type']=='Exam')]
  df_final_grade_exam=df_final_grade_exam.groupby(['id_student', 'code_module', 'code_presentation']).weighted.sum().reset_index(name='exam_grade')
  df_final=df_student.copy()
  df_final=pd.merge(df_final, df_assessments_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
  df_final=pd.merge(df_final, df_vle_sum, on=['id_student', 'code_module', 'code_presentation'], how="outer")
  df_final=pd.merge(df_final, df_vle_mean, on=['id_student', 'code_module', 'code_presentation'], how="outer")
  df_final=pd.merge(df_final, df_final_grade_cont, on=['id_student', 'code_module', 'code_presentation'], how="outer")

  df_final=pd.merge(df_final, df_final_grade_exam, on=['id_student', 'code_module', 'code_presentation'], how="outer")
  df_final=pd.merge(df_final, courses, on=['code_module', 'code_presentation'], how="outer")
  df_final['mean_clicks_per_day']=df_final['mean_clicks']/df_final['module_presentation_length']
  df_final['date_registration'] = df_final['date_registration'].fillna(0)
  df_final.loc[pd.isna(df_final["date_unregistration"]), "date_unregistration"] = df_final[pd.isna(df_final["date_unregistration"])]["module_presentation_length"].apply(
      lambda x: x
  )

  df_final['date_registration'] = df_final['date_registration'].fillna(0)
  df_final.iloc[:,17:41]=df_final.iloc[:,17:41].fillna(0)
  df_final.loc[pd.isna(df_final["exam_grade"]), "exam_grade"] = df_final[pd.isna(df_final["exam_grade"])]["code_module"].apply(condition)
  df_final.loc[pd.isna(df_final["exam_grade"]), "exam_grade"] = df_final[pd.isna(df_final["exam_grade"])]["continuous_grade"].apply(lambda x: x)
  df_final['mean_clicks_per_day'] = df_final['mean_clicks_per_day'].fillna(0)
  df_final.drop(['tma_mean_score', 'cma_mean_score', 'exam_mean_score'], axis=1, inplace=True)
  df_final['Date']=date
  df_full = pd.concat([df_full, df_final])

In [18]:
df_full.clicks_from_repeatactivity.value_counts()

0.0    32590
3.0        1
4.0        1
2.0        1
Name: clicks_from_repeatactivity, dtype: int64

In [12]:
df_full.Date.value_counts()

15    32593
30    32593
Name: Date, dtype: int64

In [13]:
#df_full.to_csv('dataset_date_cont.csv')